# Understanding the MFCAD dataset

In [1]:
from collections import Counter
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

from uvnet_project.load_mfcad import DATA_DIR, load_mfcad

## 1. Dataset directory structure

The unpacked files used in this notebook are arranged as follows:

```text
GNN-IN-CAD/
|-- data/
|   `-- MFCADDataset/
|       |-- split.json
|       |-- graph/
|       |   `-- <sample_id>.bin
|       `-- labels/
|           `-- <sample_id>_ids.json
`-- notebooks/
    `-- dataset.ipynb
```

| File or folder | What it contains |
| --- | --- |
| [split.json](../data/MFCADDataset/split.json) | Training, validation, and test sample IDs |
| [graph/](../data/MFCADDataset/graph) | One stored graph and its sampled geometry per CAD solid |
| [labels/](../data/MFCADDataset/labels) | One JSON annotation file per solid, containing its face labels and bounding boxes |

The same sample ID links the graph and annotation files. IDs already include the `_color` suffix; the label filename adds `_ids.json`.

## 2. Dataset splits and loading

### 2.1. Dataset splits

The table below shows how many CAD solids belong to the training, validation, and test splits.

In [2]:
with (DATA_DIR / "split.json").open() as file:
    splits = json.load(file)

pd.DataFrame({
    "split": list(splits),
    "number_of_solids": [len(ids) for ids in splits.values()],
})

,split,number_of_solids
0,train,9277
1,validation,3090
2,test,3094


### 2.2. Load a small training subset

Load five solids to keep the walkthrough small. `head=5` limits the number of solids loaded; omitting `head` loads the entire selected split. The overview below compares solid sizes. Class details are introduced in Section 4.

In [3]:
samples = load_mfcad("train", head=5)
print(f"Dataset: {DATA_DIR}")
print(f"Loaded {len(samples)} training solids")

summary = []
for loaded_sample in samples:
    loaded_graph = loaded_sample["graph"]
    summary.append({
        "sample_id": loaded_sample["id"],
        "faces": loaded_graph.num_nodes(),
        "stored_edges": loaded_graph.num_edges(),
    })
pd.DataFrame(summary)

Dataset: C:\Users\KaiSheng\OneDrive\Desktop\GNN-IN-CAD\data\MFCADDataset
Loaded 5 training solids


,sample_id,faces,stored_edges
0,0-0-10-12-19_color,17,84
1,1-7-7-8-13-23_color,19,102
2,3-3-4-5-5-23_color,29,162
3,1-1-5-6-13-23_color,20,108
4,1-3-5-8-14-23_color,24,132


## 3. Understand one sample

`samples` is a list of CAD solids. Each item is a dictionary describing **one whole solid**. We explicitly select the first solid and use it throughout the remaining sections.

| Dictionary key | Object | Meaning |
| --- | --- | --- |
| `id` | String | Sample ID linking the files on disk |
| `graph` | DGL graph | Faces, their connections, and attached geometry tensors |
| `faces` | List of dictionaries | One annotation per graph face, with its class and bounding box |
| `graph_metadata` | Dictionary | File-level tensor metadata returned by the graph loader; it may be empty |

The loader checks that graph-node and annotation counts agree. The list position `faces[i]` supplies the annotation for graph face `i`.

In [4]:
sample = samples[0]
graph = sample["graph"]

print("Sample ID:", sample["id"])
print("Dictionary keys:", list(sample))
print("Face annotations:", len(sample["faces"]))
print("Graph nodes:", graph.num_nodes())
print("Graph-file metadata:", sample["graph_metadata"])

Sample ID: 0-0-10-12-19_color
Dictionary keys: ['id', 'graph', 'faces', 'graph_metadata']
Face annotations: 17
Graph nodes: 17
Graph-file metadata: {}


## 4. Face annotations and machining classes

### 4.1. Inspect one face annotation

Each annotation contains `segment` (the class ID/name) and `bbox` (the XYZ minimum/maximum bounds). The bounding box describes spatial bounds; it is not a surface-area measurement. Face indices are zero-based positions in the annotation list.

In [5]:
preview_face_index = 0
print(f"Annotation for face {preview_face_index}:")
print(json.dumps(sample["faces"][preview_face_index], indent=2))

Annotation for face 0:
{
  "bbox": {
    "xmin": -1.0,
    "ymin": -0.5325959886792,
    "zmin": -1.0,
    "xmax": -1.0,
    "ymax": 1.0,
    "zmax": 1.0
  },
  "segment": {
    "name": "stock",
    "index": 15
  }
}


### 4.2. Inspect face labels

The prediction task assigns **one class to each face**. Several faces can share a class; a shared class ID does not identify a separate feature instance. `stock` is the stock/background class.

First inspect the labels of our selected solid, then the class counts across the five loaded solids.

In [6]:
face_labels = pd.DataFrame([
    face["segment"] for face in sample["faces"]
]).rename(columns={"index": "class_id", "name": "class_name"})
face_labels.index.name = "face_index"
display(face_labels[["class_id", "class_name"]])

class_count_rows = []
for loaded_sample in samples:
    counts = Counter(face["segment"]["name"] for face in loaded_sample["faces"])
    for class_name, face_count in sorted(counts.items()):
        class_count_rows.append({
            "sample_id": loaded_sample["id"], "class_name": class_name, "faces": face_count,
        })
print("Class counts in the loaded subset:")
pd.DataFrame(class_count_rows)

,class_id,class_name
face_index,,
0,15,stock
1,15,stock
2,0,rectangular_through_slot
3,15,stock
4,15,stock
5,0,rectangular_through_slot
6,15,stock
7,12,triangular_pocket
8,12,triangular_pocket


Class counts in the loaded subset:


,sample_id,class_name,faces
0,0-0-10-12-19_color,rectangular_blind_slot,5
1,0-0-10-12-19_color,rectangular_through_slot,2
2,0-0-10-12-19_color,stock,6
3,0-0-10-12-19_color,triangular_pocket,4
4,1-7-7-8-13-23_color,6sides_pocket,2
5,1-7-7-8-13-23_color,rectangular_blind_step,2
6,1-7-7-8-13-23_color,slanted_through_step,6
7,1-7-7-8-13-23_color,stock,6
8,1-7-7-8-13-23_color,triangular_through_slot,3
9,3-3-4-5-5-23_color,6sides_passage,2


### 4.3. List all classes

Our example solid contains only some of the dataset's **16 classes**. The next table collects all released class-name/ID pairs from annotation files, including stock, without loading the other graph tensors.

In [6]:
# Collect the released class catalogue beyond our five example solids.

class_pairs = set()
for label_path in sorted((DATA_DIR / "labels").glob("*_ids.json")):
    with label_path.open(encoding="utf-8") as label_file:
        annotation_faces = json.load(label_file)["body"]["faces"]
    class_pairs.update(
        (face["segment"]["index"], face["segment"]["name"])
        for face in annotation_faces
    )

machining_classes = pd.DataFrame(
    sorted(class_pairs), columns=["class_id", "class_name"]
)
print(f"Total classes (including stock): {len(machining_classes)}")
machining_classes


Total classes (including stock): 16


,class_id,class_name
0,0,rectangular_through_slot
1,1,triangular_through_slot
2,2,rectangular_passage
3,3,triangular_passage
4,4,6sides_passage
5,5,rectangular_through_step
6,6,2sides_through_step
7,7,slanted_through_step
8,8,rectangular_blind_step
9,9,triangular_blind_step


## 5. From the CAD solid to a graph

### 5.1. Map faces and shared boundaries

A solid's boundary is made of **faces** joined along **boundary curves**:

1. Create one graph node for each face.
2. Connect two nodes when their faces share a boundary curve.
3. Attach the face and curve geometry to those nodes and edges. Sections 6 and 7 explain these features.

Each face has the same number and color in both views below. The red boundary on the solid corresponds to the red connection in the graph. Colors identify individual faces.

The surfaces are approximated from the stored samples. The dataset already supplies the CAD topology; connections are not inferred from XYZ distances. The graph drawing combines reciprocal and repeated connections, and its layout has no physical spatial meaning.

**Drag the 3D model to rotate, scroll to zoom, and hover to inspect a face.**

In [8]:
# Read the stored face connections and choose one boundary to highlight.
source, destination = graph.edges(order="eid")
edge_id = 0  # Change this to inspect another stored boundary.
u, v = int(source[edge_id]), int(destination[edge_id])
face_graph = nx.Graph()
face_graph.add_nodes_from(range(graph.num_nodes()))
face_graph.add_edges_from(zip(source.tolist(), destination.tolist()))
rgba = plt.get_cmap("tab20", graph.num_nodes())(np.arange(graph.num_nodes()))
colors = [f"rgb({int(r * 255)},{int(g * 255)},{int(b * 255)})" for r, g, b, _ in rgba]
face_data = graph.ndata["x"].numpy()
curve_data = graph.edata["x"].numpy()

fig = make_subplots(
    rows=1, cols=2, specs=[[{"type": "scene"}, {"type": "xy"}]],
    column_widths=[0.55, 0.45], horizontal_spacing=0.04,
    subplot_titles=["Drag to rotate the 3D model", "Face adjacency graph"],
)

# Each face keeps the same color in the 3D model and graph.
centers = []
for i, face in enumerate(face_data):
    valid = face[..., 6] == 1
    xyz = face[..., :3].copy()
    xyz[~valid] = np.nan
    centers.append(face[..., :3][valid].mean(axis=0))
    fig.add_surface(
        x=xyz[..., 0], y=xyz[..., 1], z=xyz[..., 2],
        colorscale=[[0, colors[i]], [1, colors[i]]],
        opacity=0.65, showscale=False, connectgaps=False,
        hovertemplate=f"Face {i}<br>{face_labels.loc[i, 'class_name']}<extra></extra>",
        row=1, col=1,
    )
centers = np.array(centers)
fig.add_scatter3d(x=centers[:, 0], y=centers[:, 1], z=centers[:, 2],
                  mode="text", text=list(range(graph.num_nodes())),
                  textfont=dict(color="black", size=12), hoverinfo="skip", row=1, col=1)

# NaN rows separate curves so unrelated boundaries are not joined.
outlines = np.concatenate([np.vstack([curve[:, :3], [np.nan] * 3]) for curve in curve_data])
fig.add_scatter3d(x=outlines[:, 0], y=outlines[:, 1], z=outlines[:, 2],
                  mode="lines", line=dict(color="dimgray", width=2),
                  hoverinfo="skip", row=1, col=1)
boundary = curve_data[edge_id, :, :3]
fig.add_scatter3d(x=boundary[:, 0], y=boundary[:, 1], z=boundary[:, 2],
                  mode="lines", line=dict(color="crimson", width=8),
                  hovertemplate=f"Shared boundary: faces {u} and {v}<extra></extra>", row=1, col=1)

positions = nx.spring_layout(face_graph, seed=42)
edge_x, edge_y = [], []
for a, b in face_graph.edges():
    edge_x.extend([positions[a][0], positions[b][0], None])
    edge_y.extend([positions[a][1], positions[b][1], None])
fig.add_scatter(x=edge_x, y=edge_y, mode="lines", line=dict(color="lightgray", width=1.5),
                hoverinfo="skip", row=1, col=2)
fig.add_scatter(x=[positions[u][0], positions[v][0]], y=[positions[u][1], positions[v][1]],
                mode="lines", line=dict(color="crimson", width=4), hoverinfo="skip", row=1, col=2)
node_xy = np.array([positions[i] for i in range(graph.num_nodes())])
fig.add_scatter(x=node_xy[:, 0], y=node_xy[:, 1], mode="markers+text",
                marker=dict(size=26, color=colors), text=list(range(graph.num_nodes())),
                textfont=dict(color="black"), hovertemplate="Face %{text}<extra></extra>", row=1, col=2)

fig.update_layout(
    title=f"Sample {sample['id']} | highlighted boundary: face {u} -- face {v}",
    height=560, showlegend=False, margin=dict(l=0, r=0, t=80, b=10),
    scene=dict(aspectmode="data", dragmode="orbit",
               camera=dict(eye=dict(x=1.5, y=-1.7, z=1.2)),
               xaxis_title="X", yaxis_title="Y", zaxis_title="Z"),
)
fig.update_xaxes(visible=False, fixedrange=True)
fig.update_yaxes(visible=False, fixedrange=True, scaleanchor="x", scaleratio=1)
fig.show(renderer="plotly_mimetype", config={"scrollZoom": True, "displaylogo": False})

print(f"Faces {u} and {v} share the highlighted boundary, so nodes {u} and {v} are connected.")
print(f"{graph.num_nodes()} faces become {face_graph.number_of_nodes()} nodes.")
print(f"Stored graph: {graph.num_edges()} directed entries; "
      f"simplified drawing: {face_graph.number_of_edges()} connections.")

Faces 0 and 1 share the highlighted boundary, so nodes 0 and 1 are connected.
17 faces become 17 nodes.
Stored graph: 84 directed entries; simplified drawing: 41 connections.


### 5.2. Identify stored graph edges

Each stored edge ID has a source-face and destination-face index. The first few connections are shown below. Reciprocal entries and multiple boundaries can join the same pair of faces, so the stored edge count can exceed the number of lines in the simple drawing.

Graph/annotation face indices are not verified STEP-viewer face IDs. Mapping to an original CAD viewer requires a separate check.

In [9]:
edge_count = min(5, graph.num_edges())
pd.DataFrame({
    "edge_id": range(edge_count),
    "source_face": source[:edge_count].tolist(),
    "destination_face": destination[:edge_count].tolist(),
})

,edge_id,source_face,destination_face
0,0,0,1
1,1,0,3
2,2,0,5
3,3,0,4
4,4,0,2


## 6. Node features: sample a face surface

### 6.1. Understand the surface grid and its channels

Each graph node represents a whole face. Its geometry is sampled on a **10 by 10 grid** in the surface's two parameters, `(u, v)`. The published sampler chooses ten evenly spaced values along each parameter interval, including both limits.

| Channels | Values | Meaning |
| --- | --- | --- |
| 0-2 | `x, y, z` | Position on the underlying surface |
| 3-5 | `nx, ny, nz` | Surface-normal direction, perpendicular to the surface |
| 6 | `mask` | 1 inside or on the trimmed face; 0 outside |

The full tensor `graph.ndata["x"]` has shape `(N, 10, 10, 7)` for `N` faces. One face therefore stores 100 rows of `[x, y, z, nx, ny, nz, mask]`. These samples are attributes of one node, not 100 extra graph nodes. They are not RGB pixels.

Sections 6 and 7 use the **same MFCAD solid selected in Section 3**. Both models are rendered from its stored surface and curve grids. Here we inspect a face incident to the red boundary selected in Section 5.

In [10]:
# Share one loaded MFCAD sample across Sections 6 and 7.
sampling_sample = sample
sampling_sample_id = sampling_sample["id"]
sampling_graph = sampling_sample["graph"]
sampling_face_data = sampling_graph.ndata["x"].numpy()
sampling_curve_data = sampling_graph.edata["x"].numpy()
sampling_source, sampling_destination = sampling_graph.edges(order="eid")
sampling_edge_id = edge_id
sampling_face_index = int(sampling_destination[sampling_edge_id])
sampling_outlines = np.concatenate([
    np.vstack([curve[:, :3], [np.nan] * 3]) for curve in sampling_curve_data
])
face_grid = sampling_face_data[sampling_face_index]
face_mask = face_grid[..., 6] == 1
face_xyz = face_grid[..., :3]
uv_indices = np.stack(np.indices(face_mask.shape), axis=-1)
grid_i, grid_j = np.argwhere(face_mask)[0]

print("MFCAD sample used in Sections 6 and 7:", sampling_sample_id)
print("Graph file:", DATA_DIR / "graph" / f"{sampling_sample_id}.bin")
print("All node features:", tuple(sampling_graph.ndata["x"].shape))
print(f"Face {sampling_face_index} grid:", face_grid.shape)
print(f"One valid grid row at index ({grid_i}, {grid_j}):")
pd.DataFrame(
    [face_grid[grid_i, grid_j]], columns=["x", "y", "z", "nx", "ny", "nz", "mask"]
).round(4)

MFCAD sample used in Sections 6 and 7: 0-0-10-12-19_color
Graph file: C:\Users\KaiSheng\OneDrive\Desktop\GNN-IN-CAD\data\MFCADDataset\graph\0-0-10-12-19_color.bin
All node features: (17, 10, 10, 7)
Face 1 grid: (10, 10, 7)
One valid grid row at index (0, 3):


,x,y,z,nx,ny,nz,mask
0,-1.0,-0.3333,1.0,0.0,0.0,1.0,1.0


### 6.2. Inspect face sampling in 3D

Blue dots are valid samples; red crosses are samples outside the trimmed face. Gray lines reveal the grid's row/column structure, while orange arrows show a few normal directions. The translucent solid and its boundary outlines provide context.

**Drag to rotate and scroll to zoom.** Arrow lengths are shortened for display; directions come from the stored normals.

In [11]:
node_feature_fig = go.Figure()

# Show a translucent solid around the selected face's sample grid.
for face in sampling_face_data:
    xyz = face[..., :3].copy()
    xyz[face[..., 6] == 0] = np.nan
    node_feature_fig.add_surface(
        x=xyz[..., 0], y=xyz[..., 1], z=xyz[..., 2],
        colorscale=[[0, "#94a3b8"], [1, "#94a3b8"]],
        opacity=0.22, showscale=False, connectgaps=False, hoverinfo="skip",
    )
# Boundary outlines make the solid's shape easier to see.
node_feature_fig.add_scatter3d(
    x=sampling_outlines[:, 0], y=sampling_outlines[:, 1], z=sampling_outlines[:, 2], mode="lines",
    line=dict(color="#64748b", width=2), opacity=0.55,
    hoverinfo="skip", showlegend=False,
)
grid_rows = list(face_grid) + list(face_grid.swapaxes(0, 1))
grid_lines = np.concatenate([np.vstack([row[:, :3], [np.nan] * 3]) for row in grid_rows])
node_feature_fig.add_scatter3d(
    x=grid_lines[:, 0], y=grid_lines[:, 1], z=grid_lines[:, 2], mode="lines",
    line=dict(color="gray", width=1), hoverinfo="skip", showlegend=False,
)
for keep, color, symbol, name in [
    (face_mask, "royalblue", "circle", "Mask = 1"),
    (~face_mask, "crimson", "x", "Mask = 0"),
]:
    points = face_xyz[keep]
    node_feature_fig.add_scatter3d(
        x=points[:, 0], y=points[:, 1], z=points[:, 2], mode="markers",
        marker=dict(color=color, size=4, symbol=symbol), name=name,
        customdata=uv_indices[keep],
        hovertemplate="Grid index (%{customdata[0]}, %{customdata[1]})"
                      "<br>XYZ: %{x:.3f}, %{y:.3f}, %{z:.3f}<extra></extra>",
    )
arrows_at = np.linspace(0, int(face_mask.sum()) - 1, 5, dtype=int)
normal_points = face_xyz[face_mask][arrows_at]
normal_vectors = face_grid[..., 3:6][face_mask][arrows_at]
node_feature_fig.add_cone(
    x=normal_points[:, 0], y=normal_points[:, 1], z=normal_points[:, 2],
    u=normal_vectors[:, 0], v=normal_vectors[:, 1], w=normal_vectors[:, 2],
    anchor="tail", sizemode="absolute", sizeref=0.15,
    colorscale=[[0, "darkorange"], [1, "darkorange"]], showscale=False, hoverinfo="skip",
)
node_feature_fig.update_layout(
    title=f"MFCAD sample {sampling_sample_id}<br><sup>Face {sampling_face_index}: 100 UV samples and normal directions</sup>",
    meta=dict(sample_id=sampling_sample_id, face_index=sampling_face_index),
    height=540, margin=dict(l=0, r=0, t=85, b=20), legend=dict(orientation="h"),
    scene=dict(aspectmode="data", dragmode="orbit", camera=fig.layout.scene.camera,
               xaxis_title="X", yaxis_title="Y", zaxis_title="Z"),
)
node_feature_fig.show(renderer="plotly_mimetype", config={"scrollZoom": True, "displaylogo": False})
print(f"{int(face_mask.sum())} valid + {int((~face_mask).sum())} masked-out positions "
      f"= {face_mask.size} samples on one graph node.")

78 valid + 22 masked-out positions = 100 samples on one graph node.


## 7. Edge features: sample a boundary curve

### 7.1. Understand the curve samples and their channels

A boundary curve needs one parameter, `t`. The published sampler chooses **ten evenly spaced parameter values**, including both endpoints. Equal parameter steps need not give equal physical distances.

| Channels | Values | Meaning |
| --- | --- | --- |
| 0-2 | `x, y, z` | Position on the boundary curve |
| 3-5 | `tx, ty, tz` | Tangent direction along the curve |

The full tensor `graph.edata["x"]` has shape `(E, 10, 6)` for `E` stored directed edges. Each edge stores ten rows of `[x, y, z, tx, ty, tz]`; it has no trimming-mask channel. Row `e` is aligned with `graph.edges(order="eid")` entry `e`.

This section uses the **same loaded MFCAD solid as Section 6**. Its sample ID is repeated in both plot titles. We inspect that solid's red boundary selected in Section 5.

In [12]:
# Use the same graph, boundary ID, and raw feature arrays established in Section 6.
edge_grid = sampling_curve_data[sampling_edge_id]

print("MFCAD sample (same as Section 6):", sampling_sample_id)
print("All edge features:", tuple(sampling_graph.edata["x"].shape))
print(f"Edge {sampling_edge_id} connects faces "
      f"{int(sampling_source[sampling_edge_id])} and {int(sampling_destination[sampling_edge_id])}")
print(f"Edge {sampling_edge_id} grid:", edge_grid.shape)
print("Curve sample 0:")
pd.DataFrame(
    [edge_grid[0]], columns=["x", "y", "z", "tx", "ty", "tz"]
).round(4)

MFCAD sample (same as Section 6): 0-0-10-12-19_color
All edge features: (84, 10, 6)
Edge 0 connects faces 0 and 1
Edge 0 grid: (10, 6)
Curve sample 0:


,x,y,z,tx,ty,tz
0,-1.0,-0.5326,1.0,0.0,1.0,0.0


### 7.2. Inspect curve sampling in 3D

Red points numbered 0-9 show the stored curve positions. The connecting line approximates the curve between samples; green arrows show a few stored tangent directions. The translucent solid and its boundary outlines show where this curve sits.

**Drag to rotate and scroll to zoom.** Arrow lengths are shortened for display. Together, the face grids, curve grids, and graph connections provide the geometry and topology used to predict one class per face.

In [13]:
edge_feature_fig = go.Figure()

# Keep the same solid context and camera as the node-feature view.
for face in sampling_face_data:
    xyz = face[..., :3].copy()
    xyz[face[..., 6] == 0] = np.nan
    edge_feature_fig.add_surface(
        x=xyz[..., 0], y=xyz[..., 1], z=xyz[..., 2],
        colorscale=[[0, "#94a3b8"], [1, "#94a3b8"]],
        opacity=0.22, showscale=False, connectgaps=False, hoverinfo="skip",
    )
# Boundary outlines make the solid's shape easier to see.
edge_feature_fig.add_scatter3d(
    x=sampling_outlines[:, 0], y=sampling_outlines[:, 1], z=sampling_outlines[:, 2], mode="lines",
    line=dict(color="#64748b", width=2), opacity=0.55,
    hoverinfo="skip", showlegend=False,
)
edge_feature_fig.add_scatter3d(
    x=edge_grid[:, 0], y=edge_grid[:, 1], z=edge_grid[:, 2], mode="lines+markers+text",
    marker=dict(color="crimson", size=5), line=dict(color="crimson", width=4),
    text=list(range(len(edge_grid))), textposition="top center", name="Curve samples 0-9",
    hovertemplate="Curve sample %{text}<br>XYZ: %{x:.3f}, %{y:.3f}, %{z:.3f}<extra></extra>",
)
tangent_samples = edge_grid[::2]
edge_feature_fig.add_cone(
    x=tangent_samples[:, 0], y=tangent_samples[:, 1], z=tangent_samples[:, 2],
    u=tangent_samples[:, 3], v=tangent_samples[:, 4], w=tangent_samples[:, 5],
    anchor="tail", sizemode="absolute", sizeref=0.15,
    colorscale=[[0, "seagreen"], [1, "seagreen"]], showscale=False, hoverinfo="skip",
)
edge_feature_fig.update_layout(
    title=f"MFCAD sample {sampling_sample_id}<br><sup>Edge {sampling_edge_id}: 10 curve samples and tangent directions</sup>",
    meta=dict(sample_id=sampling_sample_id, edge_id=sampling_edge_id),
    height=540, showlegend=False, margin=dict(l=0, r=0, t=85, b=20),
    scene=dict(aspectmode="data", dragmode="orbit", camera=fig.layout.scene.camera,
               xaxis_title="X", yaxis_title="Y", zaxis_title="Z"),
)
edge_feature_fig.show(renderer="plotly_mimetype", config={"scrollZoom": True, "displaylogo": False})
print(f"{len(edge_grid)} position/tangent samples on one stored graph edge.")

10 position/tangent samples on one stored graph edge.
